### Goal: build a sentiment classifier for customer feedback. <br>
Data Source:<br> generate a synthetic feedback dataset.<br>

Build a preprocessing pipeline for:(lowercase → punctuation → tokenize → stopwords → lemmatize). 
Raw text in, clean tokens out.

In [1]:
# Step 1: Synthetic dataset

import random
import pandas as pd 

random.seed(42)

positive_template = [
    "I absolutely LOVED the {product}!! Best purchase this year.",
    "The {product} exceeded my expectations, really well made.",
    "Great value... the {product} works perfectly, 10/10 would recommend!",
    "My family is so happy with the {product}. It's been amazing!",
    "Honestly, the {product} is fantastic & the delivery was quick.",
]

negative_template = [
    "The {product} broke after TWO days. Total waste of money!!!",
    "Very disappointed... the {product} is nothing like the pictures.",
    "Worst {product} I have ever bought, do NOT recommend.",
    "The {product} arrived damaged & customer service ignored me.",
    "I regret buying this {product}. Cheap materials, terrible quality.",
]

products = ["blender", "headphones", "backpack", "kettle", "desk lamp",
            "phone case", "water bottle", "keyboard"]

rows = []
for _ in range(150):
    if random.random() < 0.5:
        text = random.choice(positive_template).format(product=random.choice(products))
        label = "positive"
    else:
        text = random.choice(negative_template).format(product=random.choice(products))
        label = "negative"
    rows.append({"review": text, "sentiment": label})

df = pd.DataFrame(rows)
df.head()

,review,sentiment
0,The desk lamp broke after TWO days. Total wast...,negative
1,"The headphones exceeded my expectations, reall...",positive
2,I regret buying this headphones. Cheap materia...,negative
3,The blender broke after TWO days. Total waste ...,negative
4,"The blender exceeded my expectations, really w...",positive


Preprocessing pipeline. Stage 1–3:<br> - normalize case,<br> - strip punctuation,<br> - tokenize

In [3]:
import string
import nltk
from nltk.tokenize import word_tokenize

# One-time downloads: NLTK ships without its data packs to stay lightweight
nltk.download("punkt")
nltk.download("punkt_tab")

def basic_clean(text):
    text = text.lower()     #STAGE 1
    text = text.translate(str.maketrans("", "", string.punctuation)) #STAGE 2
    tokens = word_tokenize(text)    #STAGE 3
    return tokens

# Review one before touching the whole DataFrame.
sample = df["review"].iloc[0]
print(sample)
print(basic_clean(sample))

[nltk_data] Downloading package punkt to /home/miringu/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/miringu/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


The desk lamp broke after TWO days. Total waste of money!!!
['the', 'desk', 'lamp', 'broke', 'after', 'two', 'days', 'total', 'waste', 'of', 'money']


Preprocessing pipeline. Stage 4-5:<br> - remove stopwords without destroying negation,<br> - lemmatize what remains,<br>

In [15]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")


def preprocess(text):
    token_list = basic_clean(text)

    #STAGE 4
    stop_words = set(stopwords.words("english"))
    negations = {"not", "no", "nor", "don", "dont", "isn", "isnt", "wasn", "wasnt"}
    cust_words = stop_words - negations

    filtered_tokens = [t for t in token_list if t not in cust_words]

    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(word) for word in filtered_tokens]

df["tokens"] = df["review"].apply(preprocess)

df[["review", "tokens"]].head(9)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/miringu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/miringu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,review,tokens
0,The desk lamp broke after TWO days. Total wast...,"[desk, lamp, broke, two, day, total, waste, mo..."
1,"The headphones exceeded my expectations, reall...","[headphone, exceeded, expectation, really, wel..."
2,I regret buying this headphones. Cheap materia...,"[regret, buying, headphone, cheap, material, t..."
3,The blender broke after TWO days. Total waste ...,"[blender, broke, two, day, total, waste, money]"
4,"The blender exceeded my expectations, really w...","[blender, exceeded, expectation, really, well,..."
5,I regret buying this water bottle. Cheap mater...,"[regret, buying, water, bottle, cheap, materia..."
6,"Honestly, the desk lamp is fantastic & the del...","[honestly, desk, lamp, fantastic, delivery, qu..."
7,The backpack broke after TWO days. Total waste...,"[backpack, broke, two, day, total, waste, money]"
8,"Worst desk lamp I have ever bought, do NOT rec...","[worst, desk, lamp, ever, bought, not, recommend]"
